# 16.3 — BCG attempts 1 and 2 on the v08 IID Pearson axis

`e22_batch_correction.ipynb` (Josh, last updated Aug 3 / files dated Aug 5) is the
batch-correction decision. scANVI is the method we pursue. The objects to use are
**not** `bcg_mouse_aligned_05*.h5ad`.

| | Attempt 1 | Attempt 2 |
|---|---|---|
| Expression | true raw UMIs | August scANVI `layers['counts']` (atlas-posed, count-scale) |
| Then | project → `normalize_total(1e4)` → `log1p` | same |
| Gene axis | **v08 IID Pearson** 1,000 ENSG | same |

Do **not** feed `counts_original` if you mean attempt 2 — that is attempt 1 again,
and on these files it is already subset to 685 genes.

## What e22 actually wrote (verified on disk)

The comparison in e22 is framed as "fixed 1,000 v07 Pearson ENSG". The **saved**
scANVI objects are a further Seurat-HVG subset of that list:

- `.../tb/data/mouse_bcg/bcg_control_scanvi_080526.h5ad` — 412 LT-HSC, **685** ENSG
- `.../tb/data/mouse_bcg/bcg_treated_scanvi_080526.h5ad` — 994 LT-HSC, same 685 ENSG
- Those 685 are a subset of v07 Pearson (685 / 1000)
- Overlap with v08 Pearson is **498 / 1000** (187 of the 685 are v07-only)
- `layers['counts_original']` = integer UMIs on those 685 genes
- `layers['counts']` = `freq * library_size` (continuous, atlas batch, same per-cell totals)
- `.X` is log-like leftover; ignore it
- Same 1,406 barcodes as the May `bcg_mouse_aligned_051026.h5ad` object

Attempt 1 therefore uses the May object's `layers['counts']` (10,866 mouse genes,
integer UMIs, same cells) so we are not stuck at 685 before the v08 projection.
Attempt 2 uses the August `layers['counts']` and can recover at most 498 v08 genes.

scANVI reference was atlas **LT-HSC + MPP3 + MPP4** only. These files are BCG LT-HSCs,
not all BCG cells.


In [1]:
import os
import sys

import h5py
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse

sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")
from speciesot_helpers import strip_ensembl_gene_id

BASE_DIR = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
JOSH_BCG = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data"
DATASET_DIR = os.path.join(BASE_DIR, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg")
ORTHO_CACHE = os.path.join(BASE_DIR, "scripts/.biomart_ortholog_cache.csv")
OUT_DIR = os.path.join(BASE_DIR, "speciesOT/baseline/analysis/bcg_mouse_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

V08_PEARSON = os.path.join(DATASET_DIR, "hvg_pearson_residuals_a_uncapped_v08.h5ad")
V07_PEARSON = os.path.join(DATASET_DIR, "hvg_pearson_residuals_atlas_full_v07.h5ad")
MAY_BCG = os.path.join(JOSH_BCG, "bcg_mouse_aligned_051026.h5ad")
AUG_CTRL = os.path.join(JOSH_BCG, "mouse_bcg/bcg_control_scanvi_080526.h5ad")
AUG_TRT = os.path.join(JOSH_BCG, "mouse_bcg/bcg_treated_scanvi_080526.h5ad")

ATTEMPT1_OUT = os.path.join(DATASET_DIR, "bcg_mouse_attempt1_rawumi_v08_iid_pearson.h5ad")
ATTEMPT2_OUT = os.path.join(DATASET_DIR, "bcg_mouse_attempt2_scanvi0805_v08_iid_pearson.h5ad")
COVERAGE_CSV = os.path.join(OUT_DIR, "bcg_v08_iid_attempts_coverage.csv")


def h5ad_var_names(path):
    with h5py.File(path, "r") as f:
        g = f["var"]
        key = g.attrs.get("_index", "index")
        key = key.decode() if isinstance(key, bytes) else key
        return [strip_ensembl_gene_id(v.decode() if isinstance(v, bytes) else str(v)) for v in g[key][:]]


v08_genes = h5ad_var_names(V08_PEARSON)
v07_genes = h5ad_var_names(V07_PEARSON)
print("v08 Pearson", len(v08_genes), "v07 Pearson", len(v07_genes))
print("axis overlap v08 ∩ v07", len(set(v08_genes) & set(v07_genes)), "/ 1000")
for p in (MAY_BCG, AUG_CTRL, AUG_TRT, ORTHO_CACHE):
    print("exists" if os.path.exists(p) else "MISSING", p)


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


v08 Pearson 1000 v07 Pearson 1000
axis overlap v08 ∩ v07 721 / 1000
exists /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/bcg_mouse_aligned_051026.h5ad
exists /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/mouse_bcg/bcg_control_scanvi_080526.h5ad
exists /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/mouse_bcg/bcg_treated_scanvi_080526.h5ad
exists /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/.biomart_ortholog_cache.csv


## 1. Inspect the August scANVI objects


In [2]:
def _layer(a, name):
    X = a.layers[name]
    return X.toarray() if sp_sparse.issparse(X) else np.asarray(X)


ctrl = sc.read_h5ad(AUG_CTRL)
trt = sc.read_h5ad(AUG_TRT)
print("control", ctrl.shape, "treated", trt.shape)
print("layers", list(ctrl.layers.keys()), "obs sample", list(ctrl.obs.columns))
print("var sample", list(ctrl.var_names[:5]))
assert list(ctrl.var_names.astype(str)) == list(trt.var_names.astype(str))

aug_genes = [strip_ensembl_gene_id(g) for g in ctrl.var_names.astype(str)]
print(f"August n_vars={len(aug_genes)}  subset of v07? {set(aug_genes) <= set(v07_genes)}")
print(f"August ∩ v08 Pearson: {len(set(aug_genes) & set(v08_genes))} / 1000")
print(f"August genes not in v08: {len(set(aug_genes) - set(v08_genes))}")

for label, a in (("control", ctrl), ("treated", trt)):
    orig = _layer(a, "counts_original")
    posed = _layer(a, "counts")
    print(
        f"  {label}: counts_original integer-like={np.allclose(orig, np.round(orig))} "
        f"max={orig.max():.1f};  counts (posed) max={posed.max():.2f} "
        f"mean_orig={orig.mean():.3f} mean_posed={posed.mean():.3f}"
    )
    print(f"           .X max={float(np.asarray(a.X).max()):.2f} (ignore; log-like leftover)")


control (412, 685) treated (994, 685)
layers ['counts', 'counts_original'] obs sample ['n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'day', 'leiden', 'cell_type', 'shared_cell_type', 'study', 'cell_type_scanvi', '_scvi_batch', '_scvi_labels']
var sample ['ENSG00000143546', 'ENSG00000164047', 'ENSG00000148180', 'ENSG00000170323', 'ENSG00000148346']
August n_vars=685  subset of v07? True
August ∩ v08 Pearson: 498 / 1000
August genes not in v08: 187
  control: counts_original integer-like=True max=177.0;  counts (posed) max=160.65 mean_orig=2.774 mean_posed=2.774
           .X max=5.62 (ignore; log-like leftover)
  treated: counts_original integer-like=True max=401.0;  counts (posed) max=412.53 mean_orig=4.680 mean_posed=4.680
           .X max=5.18 (ignore; log-like leftover)


## 2. Ortholog map (needed only for attempt 1)

August files are already human ENSG. The May UMI object is mouse symbols + `gene_ids`.


In [3]:
ortho_df = pd.read_csv(ORTHO_CACHE)
if "orthology_type" in ortho_df.columns:
    ortho_df = ortho_df[ortho_df["orthology_type"] == "ortholog_one2one"].copy()
ensg2musg = {}
for _, row in ortho_df.iterrows():
    h = strip_ensembl_gene_id(str(row["human_ensembl_id"]))
    m = strip_ensembl_gene_id(str(row["mouse_ensembl_id"]))
    if h and m:
        ensg2musg[h] = m
print(f"one2one ENSG→ENSMUSG: {len(ensg2musg)}")

may = sc.read_h5ad(MAY_BCG)
may.obs_names_make_unique()
assert "counts" in may.layers and "gene_ids" in may.var.columns
print("May BCG", may.shape, "layers", list(may.layers.keys()))

musg2col = {}
for vi in range(may.n_vars):
    m = strip_ensembl_gene_id(str(may.var["gene_ids"].iloc[vi]))
    if m and m not in musg2col:
        musg2col[m] = vi
ensg_to_may_col = {h: musg2col[m] for h, m in ensg2musg.items() if m in musg2col}
print(f"May columns with ENSMUSG: {len(musg2col)} / {may.n_vars}")
print(f"v08 Pearson genes with a May UMI column: {sum(g in ensg_to_may_col for g in v08_genes)} / 1000")

aug_barcodes = set(ctrl.obs_names) | set(trt.obs_names)
print(f"August barcodes in May object: {sum(b in may.obs_names for b in aug_barcodes)} / {len(aug_barcodes)}")


one2one ENSG→ENSMUSG: 14451
May BCG (1406, 10866) layers ['counts']
May columns with ENSMUSG: 10866 / 10866
v08 Pearson genes with a May UMI column: 507 / 1000
August barcodes in May object: 1405 / 1405


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [4]:
def project_counts_to_v08(X_src, col_for_ensg, obs, extra_obs=None):
    """Reorder/zero-fill a count matrix onto v08 Pearson, then Scanpy normalize+log1p."""
    if sp_sparse.issparse(X_src):
        X_src = X_src.toarray()
    X_src = np.asarray(X_src, dtype=np.float32)
    X = np.zeros((X_src.shape[0], len(v08_genes)), dtype=np.float32)
    n_hit = 0
    for j, g in enumerate(v08_genes):
        c = col_for_ensg.get(g)
        if c is not None:
            X[:, j] = X_src[:, c]
            n_hit += 1
    var = pd.DataFrame(index=pd.Index(v08_genes, name="ensg"))
    var["in_source"] = [g in col_for_ensg for g in v08_genes]
    keep = [c for c in ["condition", "species", "cell_type", "study", "day"] if c in obs.columns]
    out = ad.AnnData(X=X, obs=obs[keep].copy(), var=var)
    if extra_obs:
        for k, v in extra_obs.items():
            out.obs[k] = v
    sc.pp.normalize_total(out, target_sum=1e4)
    sc.pp.log1p(out)
    out.X = np.asarray(out.X, dtype=np.float32)
    return out, n_hit


## 3. Attempt 1 — raw UMIs (May `layers['counts']`) → v08 → Scanpy

Same 1,406 LT-HSCs as e22. Full 10,866-gene UMI matrix, not the 685-gene August slice.


In [5]:
may.obs["condition"] = "mouse"
may.obs["species"] = "mouse"
a1, n1 = project_counts_to_v08(may.layers["counts"], ensg_to_may_col, may.obs)
a1.uns["attempt"] = 1
a1.uns["source"] = MAY_BCG
a1.uns["expression"] = "raw UMIs (May layers['counts'])"
a1.uns["n_shared_with_v08"] = int(n1)
a1.write_h5ad(ATTEMPT1_OUT)
print(f"attempt 1: {n1} / 1000 v08 genes present")
print(f"  wrote {ATTEMPT1_OUT}  shape={a1.shape}  .X mean={float(a1.X.mean()):.4f} max={float(a1.X.max()):.4f}")


attempt 1: 507 / 1000 v08 genes present
  wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_attempt1_rawumi_v08_iid_pearson.h5ad  shape=(1406, 1000)  .X mean=0.5008 max=6.8777


## 4. Attempt 2 — August scANVI `layers['counts']` → v08 → Scanpy

`normalize_total` + `log1p` **does** belong here: the new decode is count-scale
(`library_size=1.0` frequencies × original UMI total), not the old CP10k `.X`.


In [6]:
ctrl.obs["condition"] = "mouse"
trt.obs["condition"] = "mouse"
ctrl.obs["species"] = "mouse"
trt.obs["species"] = "mouse"
ctrl.obs["bcg_arm"] = "control"
trt.obs["bcg_arm"] = "treated"

aug = ad.concat([ctrl, trt], join="inner", index_unique=None)
aug.obs_names_make_unique()
aug_col = {strip_ensembl_gene_id(g): i for i, g in enumerate(aug.var_names.astype(str))}
a2, n2 = project_counts_to_v08(aug.layers["counts"], aug_col, aug.obs, extra_obs={"bcg_arm": aug.obs["bcg_arm"].to_numpy()})
a2.uns["attempt"] = 2
a2.uns["source"] = [AUG_CTRL, AUG_TRT]
a2.uns["expression"] = "scANVI atlas-posed counts (August layers['counts'])"
a2.uns["n_shared_with_v08"] = int(n2)
a2.write_h5ad(ATTEMPT2_OUT)
print(f"attempt 2: {n2} / 1000 v08 genes present (cap is August ∩ v08 = {len(set(aug_genes) & set(v08_genes))})")
print(f"  wrote {ATTEMPT2_OUT}  shape={a2.shape}  .X mean={float(a2.X.mean()):.4f} max={float(a2.X.max()):.4f}")
print("  bcg_arm", a2.obs["bcg_arm"].value_counts().to_dict())


attempt 2: 498 / 1000 v08 genes present (cap is August ∩ v08 = 498)
  wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_attempt2_scanvi0805_v08_iid_pearson.h5ad  shape=(1406, 1000)  .X mean=1.0429 max=6.3002
  bcg_arm {'treated': 994, 'control': 412}


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## 5. Coverage summary

16.2's 507 / 1000 was mentor `.X` on the May object (old decode), not a usable attempt.


In [7]:
rows = [
    {
        "input": "attempt 1 — May raw UMIs (10866 mouse genes)",
        "n_source_genes": may.n_vars,
        "n_shared_v08": int(n1),
        "n_zero_filled": 1000 - int(n1),
        "coverage_pct": round(100.0 * n1 / 1000, 1),
        "out": ATTEMPT1_OUT,
    },
    {
        "input": "attempt 2 — Aug scANVI posed counts (685 ENSG)",
        "n_source_genes": len(aug_genes),
        "n_shared_v08": int(n2),
        "n_zero_filled": 1000 - int(n2),
        "coverage_pct": round(100.0 * n2 / 1000, 1),
        "out": ATTEMPT2_OUT,
    },
    {
        "input": "August object vs v07 Pearson (his axis, not ours)",
        "n_source_genes": len(aug_genes),
        "n_shared_v08": len(set(aug_genes) & set(v07_genes)),
        "n_zero_filled": 1000 - len(set(aug_genes) & set(v07_genes)),
        "coverage_pct": round(100.0 * len(set(aug_genes) & set(v07_genes)) / 1000, 1),
        "out": "(no export; he already lives here)",
    },
]
cov = pd.DataFrame(rows)
cov.to_csv(COVERAGE_CSV, index=False)
print(cov.to_string(index=False))
print(f"\nwrote {COVERAGE_CSV}")
print(
    "\nTomorrow: attempt 1 file → predict_new_input.sh is WRONG "
    "(already log1p'd). Feed raw-count .h5ad to the script, or skip the script "
    "and encode these already-normalized matrices. These two files are for "
    "inspecting coverage and for a Scanpy-matched encode, not for the script's phase 1."
)


                                            input  n_source_genes  n_shared_v08  n_zero_filled  coverage_pct                                                                                                                                                    out
     attempt 1 — May raw UMIs (10866 mouse genes)           10866           507            493          50.7     /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_attempt1_rawumi_v08_iid_pearson.h5ad
   attempt 2 — Aug scANVI posed counts (685 ENSG)             685           498            502          49.8 /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_attempt2_scanvi0805_v08_iid_pearson.h5ad
August object vs v07 Pearson (his axis, not ours)             685           685            315          68.5                                                                                                                

## 6. Gene lists for the mentor (pre-HVG universe + v08 Pearson 1,000)

The v08 Pearson HVG was selected from the **14,451 one-to-one human↔mouse
orthologs** after assay filter (`matched_full` was `(89760, 14451)`). Those 14,451
are the gene list **before any HVG**. The 1,000 below are
`hvg_pearson_residuals_a_uncapped_v08.h5ad` `var_names` in model order.

Written next to the training `.h5ad` so Josh can point `ATLAS_HVG_H5AD` / a
`genes.txt` read at these paths (lab-group readable).


In [ ]:
GENE_LIST_DIR = os.path.join(DATASET_DIR, "gene_lists")
os.makedirs(GENE_LIST_DIR, exist_ok=True)

ortho = pd.read_csv(ORTHO_CACHE)
if "orthology_type" in ortho.columns:
    ortho = ortho[ortho["orthology_type"] == "ortholog_one2one"].copy()
for col in ("human_ensembl_id", "mouse_ensembl_id"):
    ortho[col] = ortho[col].astype(str).map(strip_ensembl_gene_id)
ortho = ortho.drop_duplicates("human_ensembl_id", keep="first").reset_index(drop=True)

v08 = [strip_ensembl_gene_id(g) for g in v08_genes]
assert len(v08) == 1000
assert set(v08) <= set(ortho["human_ensembl_id"]), "v08 HVG not a subset of the pre-HVG ortholog table"

PRE_HVG_TXT = os.path.join(GENE_LIST_DIR, "pre_hvg_one2one_orthologs_ensg.txt")
PRE_HVG_CSV = os.path.join(GENE_LIST_DIR, "pre_hvg_one2one_orthologs.csv")
V08_TXT = os.path.join(GENE_LIST_DIR, "hvg_pearson_residuals_a_uncapped_v08_genes.txt")
V08_CSV = os.path.join(GENE_LIST_DIR, "hvg_pearson_residuals_a_uncapped_v08_genes.csv")

with open(PRE_HVG_TXT, "w") as fh:
    fh.write("\n".join(ortho["human_ensembl_id"].tolist()) + "\n")
ortho[["human_ensembl_id", "human_gene_name", "mouse_ensembl_id", "mouse_gene_name"]].to_csv(
    PRE_HVG_CSV, index=False
)

with open(V08_TXT, "w") as fh:
    fh.write("\n".join(v08) + "\n")
sym = ortho.set_index("human_ensembl_id")
v08_df = pd.DataFrame({
    "human_ensembl_id": v08,
    "human_gene_name": [sym.loc[g, "human_gene_name"] if g in sym.index else "" for g in v08],
    "mouse_ensembl_id": [sym.loc[g, "mouse_ensembl_id"] if g in sym.index else "" for g in v08],
    "mouse_gene_name": [sym.loc[g, "mouse_gene_name"] if g in sym.index else "" for g in v08],
})
v08_df.to_csv(V08_CSV, index=False)

print("Pre-HVG (one-to-one orthologs, before any HVG):")
print(f"  n={len(ortho)}")
print(f"  {PRE_HVG_TXT}")
print(f"  {PRE_HVG_CSV}")
print("v08 Pearson HVG (best model gene axis, model order):")
print(f"  n={len(v08)}")
print(f"  {V08_TXT}")
print(f"  {V08_CSV}")
print(f"v08 ⊂ pre-HVG: {set(v08) <= set(ortho['human_ensembl_id'])}")
